# Identity Card OCR for Reliable Structured KYC Extraction

### COMPFEST 18 Data Science Academy Final Project

**Team:** Ketolak Pacil  
**Members:** Orie Abyan Maulana · Aditya Akbar · Khansa Firyal

**Objective:** mengekstraksi **Name**, **Birth Date**, dan **Address** dari dokumen identitas yang heterogen untuk mendukung proses KYC secara lebih cepat, konsisten, dan terukur.

**Final V28 Labeled Benchmark:** **66.77% Macro Field Exact Match**

**Core flow:** `Quality Assessment → Adaptive Preprocessing → PP OCRv6 Detection → SVTRv2 Recognition → Evidence Driven Routing → Specialist Field Extraction`

> **Reproducibility scope**  
> Notebook ini dapat di rerun secara ringan pada level artifact. EDA, diagnostic analysis, final metrics, visual evidence, dan competition output dibangun ulang dari frozen artifacts hasil development tim. Raw OCR model tidak di retrain dan seluruh 732 image tidak di OCR ulang di notebook submission ini.

## How to Read This Notebook

Notebook mengikuti satu alur argumentasi yang konsisten:

**Business Need → Data Evidence → Key Finding → Pipeline Response → Final Evaluation → Business Recommendation**

Tiga jenis evidence dibedakan sepanjang notebook:

- **DATA EVIDENCE** untuk analisis yang dihitung ulang dari project artifacts.
- **DEVELOPMENT EVIDENCE** untuk implementasi atau hasil yang ditelusuri dari source V3 sampai V27.
- **FINAL BENCHMARK** untuk evaluasi yang dihitung ulang dari released V28 labeled benchmark.

> **Public repository edition:** original code and narrative are retained. Embedded identity-data archives and cell outputs have been removed. Authorized evidence is loaded from `private/artifacts/`. Original executed notebooks remain in the local `private/original_final_project/` folder.


# 1 Artifact Setup and Reproducibility

Seluruh frozen OCR output, preprocessing record, routing evidence, field prediction, dan evaluation artifact yang dibaca notebook ini berasal dari **dataset resmi COMPFEST 18** dan development pipeline tim. Artifact tersebut adalah derived outputs, bukan external supporting dataset.

Pretrained OCR models digunakan sesuai challenge rules. Notebook tidak menambahkan dataset eksternal, tidak menggunakan LLM atau VLM sebagai komponen OCR, dan tidak melakukan AutoML.

In [ ]:
# Standard library
import base64
import hashlib
import io
import json
import re
import zipfile
from itertools import combinations
from pathlib import Path

# Data and numerical
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPyImage

# Lightweight diagnostic modeling
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import roc_auc_score

SEED = 42
RNG = np.random.default_rng(SEED)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 100)

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "experiments").is_dir()), Path.cwd())
WORK_DIR = PROJECT_ROOT / "private" / "notebook_runs"
ARTIFACT_DIR = PROJECT_ROOT / "private" / "artifacts"
OUTPUT_DIR = WORK_DIR / "outputs"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COLORS = {
    "navy": "#16324F",
    "blue": "#2F80ED",
    "cyan": "#37B7C3",
    "teal": "#2A9D8F",
    "green": "#4CAF50",
    "orange": "#F4A261",
    "coral": "#E76F51",
    "purple": "#7B61FF",
    "gray": "#9AA5B1",
    "light_gray": "#E9EEF3",
    "ink": "#263238",
}

ROUTE_COLORS = [
    COLORS["blue"], COLORS["orange"], COLORS["teal"], COLORS["purple"],
    COLORS["coral"], COLORS["cyan"], COLORS["green"], "#8D6E63",
    "#5C6BC0", "#EC407A", COLORS["gray"],
]

print("Environment ready")
print("Seed:", SEED)

In [ ]:
# Local evidence bundle; intentionally excluded from the public repository.
manifest_path = ARTIFACT_DIR / "MANIFEST.json"
if not manifest_path.exists():
    raise FileNotFoundError("Place the authorized evidence bundle under private/artifacts; see docs/data_setup.md")
with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)
print(f"Project evidence available: {len(manifest)} tracked files")


In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

integrity = pd.DataFrame([
    {
        "Artifact": item["path"],
        "Kind": item.get("kind", "artifact"),
        "Exists": (ARTIFACT_DIR / item["path"]).exists(),
        "Hash Match": (ARTIFACT_DIR / item["path"]).exists()
        and sha256_file(ARTIFACT_DIR / item["path"]) == item["sha256"],
    }
    for item in manifest
])

assert integrity["Exists"].all()
assert integrity["Hash Match"].all()

print("Artifact Integrity Check")
print("Tracked files :", len(integrity))
print("Hash matches  :", int(integrity["Hash Match"].sum()))
print("Status        : PASS")

In [ ]:
V28_DIR = ARTIFACT_DIR / "results_v28_generic_unknown_peer_consensus"
V27_DIR = ARTIFACT_DIR / "reference_outputs" / "results_v27_fuzzy_peer_medoid_names"

v28 = pd.read_csv(V28_DIR / "predictions_released.csv")
v27 = pd.read_csv(V27_DIR / "predictions_released.csv")
official_test = pd.read_csv(ARTIFACT_DIR / "results" / "test_predictions.csv")
field_proposals = pd.read_csv(ARTIFACT_DIR / "results_v17_per_layout_generic_mrz" / "field_proposals.csv")
generic_unknown_audit = pd.read_csv(ARTIFACT_DIR / "reports" / "GENERIC_ID_UNKNOWN_ROUTE_AUDIT.csv")
development_timeline = pd.read_csv(ARTIFACT_DIR / "development_timeline.csv")

with open(V28_DIR / "metrics.json", "r", encoding="utf-8") as f:
    v28_metrics_ref = json.load(f)
with open(V27_DIR / "metrics.json", "r", encoding="utf-8") as f:
    v27_metrics_ref = json.load(f)
with open(V28_DIR / "validation.json", "r", encoding="utf-8") as f:
    v28_validation = json.load(f)

labeled = v28[v28["has_ground_truth"].astype(bool)].copy().reset_index(drop=True)
test_artifact_rows = v28[~v28["has_ground_truth"].astype(bool)].copy().reset_index(drop=True)

# Detailed route used only as pipeline assisted diagnostic evidence.
proposal_route = field_proposals.set_index("filename")["route"].to_dict()
route_map = {
    "MRZ_ICAO": "MRZ / Passport",
    "EU_DRIVING_LICENSE": "EU Driving Licence",
    "CHILE_ID": "Chile ID",
    "CZECH_ID": "Czech ID",
    "UKRAINE_PASSPORT": "Ukraine Passport",
    "PORTUGAL_CITIZEN_CARD": "Portugal Citizen Card",
    "HK_MAINLAND_PERMIT": "HK Mainland Permit",
    "SPAIN_ID": "Spain ID",
    "TURKEY_ID": "Turkey ID",
}

def detailed_route(row):
    if str(row["document_family"]).upper() == "MYKAD":
        return "MyKad"
    if row["filename"] in proposal_route:
        return route_map.get(proposal_route[row["filename"]], proposal_route[row["filename"]])
    return "Unknown Route"

labeled["route_final"] = labeled.apply(detailed_route, axis=1)

print("Artifact Loading Summary")
print("V28 released rows      :", len(v28))
print("Labeled benchmark rows :", len(labeled))
print("Test artifact rows     :", len(test_artifact_rows))
print("Official test rows     :", len(official_test))

### Provenance Check

Artifact loading dibuat eksplisit agar angka di notebook tidak muncul sebagai hard coded result. Statistik, EDA, final metrics, dan output submission dihitung ulang dari tables yang baru saja dibaca.

Frozen visual examples juga disimpan sebagai project derived evidence dari original development run. Hal ini memungkinkan notebook tetap visual dan rerunnable tanpa menjalankan heavy OCR lagi.

# 2 Business Understanding

Dokumen identitas seperti KTP, paspor, dan SIM merupakan sumber informasi penting dalam berbagai proses bisnis yang membutuhkan data identitas secara terstruktur. Informasi seperti **Name, Date of Birth atau DOB, dan Address** menjadi input utama pada proses identifikasi dan verifikasi identitas dalam KYC.

Ketika jumlah dokumen meningkat, ekstraksi manual menjadi kurang efisien karena membutuhkan waktu dan sumber daya yang lebih besar. Selain itu, human reading error pada nama, tanggal lahir, atau alamat dapat membuat data yang diteruskan ke proses berikutnya menjadi tidak akurat.

Tantangan menjadi lebih besar karena dokumen pada dataset memiliki struktur, negara asal, kondisi pencahayaan, ketajaman, dan kualitas capture yang berbeda. Dataset juga mengandung repeated captures dalam label derived identity groups. Capture yang merepresentasikan group yang sama dapat memiliki kualitas visual dan hasil OCR yang berbeda.

Karena itu, dua aspek yang penting adalah:

1. **Accuracy**, yaitu kemampuan sistem menghasilkan field yang sesuai dengan ground truth.
2. **Consistency**, yaitu kemampuan sistem mempertahankan informasi yang selaras ketika terdapat repeated captures dengan evidence yang berbeda.

### Business Objective

Membangun pipeline yang mampu mengekstraksi **Name, DOB, dan Address** secara otomatis dengan akurasi yang terukur, sambil tetap menyediakan evidence dan confidence yang dapat digunakan untuk menentukan apakah sebuah hasil cukup aman untuk diproses otomatis atau membutuhkan human review.

# 3 Problem Statement

Berdasarkan kebutuhan bisnis tersebut, tantangan utama bukan hanya membaca teks dari satu gambar. Sistem juga harus menangani variasi kualitas capture, struktur dokumen, field availability, dan repeated captures.

> **Bagaimana membangun pipeline OCR yang mampu menghasilkan Name, DOB, dan Address secara otomatis dari dokumen identitas yang heterogen, mempertahankan evaluasi yang fair ketika terdapat repeated captures, dan menyediakan structured output yang cukup traceable untuk mendukung proses KYC?**

Keberhasilan akan dinilai melalui kombinasi **field level Exact Match**, **Full Record Exact Match**, **Character Error Rate**, dan **Normalized Similarity**. Setiap metric menjawab pertanyaan yang berbeda sehingga interpretasi tidak bergantung pada satu angka saja.

# 4 Data Understanding and Integrity Audit

Sebelum melihat pola yang lebih kompleks, notebook memeriksa ukuran dataset, field availability, dan konsistensi label. Langkah ini juga menentukan cara kita membaca metric Address pada bagian evaluasi.

In [ ]:
overview = pd.DataFrame([
    ["Labeled images", len(labeled)],
    ["Test images", len(test_artifact_rows)],
    ["Total images", len(v28)],
    ["Target fields", 3],
    ["Unique normalized Name values", labeled["name_gt"].fillna("").astype(str).str.strip().str.upper().nunique()],
    ["Label derived identity groups", labeled["identity_group"].nunique()],
], columns=["Dataset Item", "Value"])

display(overview.style.hide(axis="index"))

In [ ]:
field_availability = pd.DataFrame([
    ["Name", labeled["name_gt"].fillna("").astype(str).str.strip().ne("").sum()],
    ["Birth Date", labeled["birth_date_gt"].fillna("").astype(str).str.strip().ne("").sum()],
    ["Address", labeled["address_present_gt"].astype(bool).sum()],
], columns=["Field", "Available Rows"])
field_availability["Missing Rows"] = len(labeled) - field_availability["Available Rows"]
field_availability["Availability"] = field_availability["Available Rows"] / len(labeled)

display(field_availability.style.format({"Availability": "{:.1%}"}).hide(axis="index"))

In [ ]:
plt.figure(figsize=(8.4, 4.8))
colors = [COLORS["blue"], COLORS["teal"], COLORS["orange"]]
bars = plt.bar(field_availability["Field"], field_availability["Available Rows"], color=colors, width=0.62)
plt.ylim(0, 690)
plt.ylabel("Labeled documents")
plt.title("Field Availability Across the Labeled Dataset")
for bar, value in zip(bars, field_availability["Available Rows"]):
    plt.text(bar.get_x() + bar.get_width()/2, value + 10, f"{int(value)}", ha="center", fontsize=10)
plt.tight_layout()
plt.show()

**Key Finding**  
Name dan Birth Date tersedia pada seluruh labeled rows, sedangkan operationally valid Address hanya tersedia pada **112 dari 632 rows**.

**Pipeline Implication**  
Address tidak dapat dievaluasi seperti field yang selalu tersedia. Karena itu, final notebook akan selalu menampilkan **all row Address EM** bersama **present only Address EM** agar hasil tidak misleading.

# 5 Repeated Captures Create Opportunity and Validation Risk

Dataset tidak terdiri dari 632 observasi yang sepenuhnya independen. Sebagian kecil label derived groups memiliki banyak capture sehingga satu identity group dapat mendominasi jumlah image.

In [ ]:
group_sizes = labeled.groupby("identity_group").size().rename("Captures").reset_index()
capture_dist = group_sizes["Captures"].value_counts().sort_index().rename_axis("Captures per Group").reset_index(name="Identity Groups")
repeated_groups = group_sizes[group_sizes["Captures"] > 1]
repeated_group_ids = set(repeated_groups["identity_group"])
repeated_image_count = int(labeled["identity_group"].isin(repeated_group_ids).sum())

print("Label derived identity groups :", len(group_sizes))
print("Repeated groups               :", len(repeated_groups))
print("Images inside repeated groups :", repeated_image_count)
print("Share of labeled images       :", f"{repeated_image_count / len(labeled):.1%}")
display(capture_dist.style.hide(axis="index"))

In [ ]:
plt.figure(figsize=(8.4, 4.8))
bar_colors = [COLORS["gray"] if x == 1 else COLORS["cyan"] if x <= 2 else COLORS["purple"] for x in capture_dist["Captures per Group"]]
plt.bar(capture_dist["Captures per Group"].astype(str), capture_dist["Identity Groups"], color=bar_colors, width=0.68)
plt.xlabel("Captures per label derived group")
plt.ylabel("Number of groups")
plt.title("Repeated Capture Structure")
plt.tight_layout()
plt.show()

### Random Split Overlap Simulation

Random image level splitting berisiko menempatkan capture dari group yang sama pada train dan validation. Notebook mensimulasikan 500 split 80:20 untuk mengukur seberapa besar repeated group overlap yang diharapkan.

In [ ]:
sim_rng = np.random.default_rng(SEED)
repeated_ids = np.array(sorted(repeated_group_ids))
leakage_trials = []
all_positions = np.arange(len(labeled))

for _ in range(500):
    train_idx = sim_rng.choice(all_positions, size=int(0.8 * len(labeled)), replace=False)
    train_set = set(train_idx.tolist())
    val_idx = np.array([i for i in all_positions if i not in train_set])
    train_groups = set(labeled.iloc[train_idx]["identity_group"])
    val_groups = set(labeled.iloc[val_idx]["identity_group"])
    overlap = len((train_groups & val_groups) & repeated_group_ids) / len(repeated_group_ids)
    leakage_trials.append(overlap)

leakage_trials = np.asarray(leakage_trials)
print("Mean repeated group overlap :", f"{leakage_trials.mean():.2%}")
print("5th percentile              :", f"{np.quantile(leakage_trials, 0.05):.2%}")
print("95th percentile             :", f"{np.quantile(leakage_trials, 0.95):.2%}")

In [ ]:
plt.figure(figsize=(8.4, 4.8))
plt.hist(100 * leakage_trials, bins=np.arange(60, 102, 2), color=COLORS["cyan"], edgecolor="white", alpha=0.88)
plt.axvline(100 * leakage_trials.mean(), color=COLORS["coral"], linewidth=2.2, linestyle=":", label=f"Mean {100*leakage_trials.mean():.1f}%")
plt.xlabel("Repeated groups present in both splits (%)")
plt.ylabel("Simulation count")
plt.title("Expected Repeated Group Overlap Under Random Image Splitting")
plt.legend(frameon=False)
plt.tight_layout()
plt.show()

**Key Finding**  
Sekitar **91.2% repeated identity groups** diperkirakan muncul pada kedua split ketika image dibagi secara random.

**Pipeline Implication**  
Ground truth identity groups layak digunakan untuk membuat component validation yang identity disjoint. Repeated captures juga memberi opportunity untuk mempelajari apakah evidence antar capture dapat saling melengkapi.

# 6 Capture Quality Is Highly Heterogeneous

Raw capture statistics menunjukkan bahwa dokumen tidak berada pada satu quality regime. Brightness, sharpness, glare, dan illumination memiliki rentang yang lebar. Ini menjadi alasan langsung mengapa satu preprocessing transform tidak digunakan untuk seluruh image.

In [ ]:
quality_cols = ["mean_luminance", "luminance_std", "blur_score", "glare_ratio", "illumination_variation", "edge_density"]
quality_summary = labeled[quality_cols].describe(percentiles=[0.05, 0.50, 0.95]).T[["min", "5%", "50%", "95%", "max"]]
display(quality_summary.round(3))

In [ ]:
plt.figure(figsize=(8.4, 4.8))
plt.hist(labeled["mean_luminance"].dropna(), bins=30, color=COLORS["orange"], edgecolor="white", alpha=0.9)
plt.xlabel("Mean luminance")
plt.ylabel("Documents")
plt.title("Brightness Distribution Across Labeled Captures")
plt.tight_layout()
plt.show()

In [ ]:
positive_blur = labeled.loc[labeled["blur_score"] > 0, "blur_score"]
plt.figure(figsize=(8.4, 4.8))
plt.hist(np.log10(positive_blur), bins=30, color=COLORS["purple"], edgecolor="white", alpha=0.88)
plt.xlabel("log10 blur score")
plt.ylabel("Documents")
plt.title("Sharpness Variation Spans Multiple Orders of Magnitude")
plt.tight_layout()
plt.show()

In [ ]:
quality_counts = labeled["quality_group"].fillna("unknown").value_counts().rename_axis("Quality Group").reset_index(name="Documents")
quality_counts["Share"] = quality_counts["Documents"] / quality_counts["Documents"].sum()
display(quality_counts.style.format({"Share": "{:.1%}"}).hide(axis="index"))

In [ ]:
quality_palette = {
    "clean": COLORS["green"],
    "glare": COLORS["orange"],
    "dark": COLORS["navy"],
    "blur": COLORS["purple"],
    "severe_blur": COLORS["coral"],
    "low_contrast": COLORS["gray"],
    "uneven_light": COLORS["cyan"],
    "unknown": COLORS["light_gray"],
}
plot_q = quality_counts.copy()
plt.figure(figsize=(9.2, 5.1))
plt.bar(plot_q["Quality Group"], plot_q["Documents"], color=[quality_palette.get(x, COLORS["blue"]) for x in plot_q["Quality Group"]])
plt.ylabel("Documents")
plt.title("Pipeline Assisted Quality Groups")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

### Representative Capture Examples

Statistical distributions perlu dilengkapi visual evidence. Gallery berikut adalah frozen visual output dari original dataset run yang memilih capture pada extreme dan median quality conditions.

In [ ]:
display(IPyImage(filename=str(ARTIFACT_DIR / "visuals" / "quality_examples.png"), width=1050))

**Key Finding**  
Dataset memiliki capture yang sangat gelap, blur, glare, dan juga image yang sudah tajam serta terang.

**Pipeline Implication**  
Preprocessing harus adaptive. Aggressive enhancement yang diterapkan ke semua dokumen dapat membantu satu quality group tetapi justru merusak image yang sudah baik.

# 7 Document Structure Changes the Extraction Problem

OCR text tidak memiliki arti yang sama pada semua dokumen. Position, field label, numbering convention, MRZ availability, dan expected Address berubah menurut document family. Karena itu, document routing bukan cosmetic stage tetapi bagian inti dari structured extraction.

In [ ]:
route_counts = labeled["route_final"].value_counts().rename_axis("Route").reset_index(name="Documents")
route_counts["Share"] = route_counts["Documents"] / len(labeled)
display(route_counts.style.format({"Share": "{:.1%}"}).hide(axis="index"))

In [ ]:
plot_routes = route_counts.sort_values("Documents")
color_lookup = {route: ROUTE_COLORS[i % len(ROUTE_COLORS)] for i, route in enumerate(route_counts["Route"])}
plt.figure(figsize=(9.2, 6.0))
plt.barh(plot_routes["Route"], plot_routes["Documents"], color=[color_lookup[x] for x in plot_routes["Route"]])
plt.xlabel("Labeled documents")
plt.title("Document Route Distribution")
plt.tight_layout()
plt.show()

### Representative Document Families

Frozen visual evidence di bawah menunjukkan bahwa heterogeneity bukan sekadar numerical label. Struktur passport, MyKad, driving licence, country specific ID, dan unknown layout memang berbeda secara visual.

In [ ]:
display(IPyImage(filename=str(ARTIFACT_DIR / "visuals" / "route_examples.png"), width=1050))

### Address Availability by Route

Field availability juga berubah menurut document route. Ini penting karena parser tidak seharusnya memaksa Address pada dokumen yang memang tidak menyediakannya.

In [ ]:
address_by_route = (
    labeled.groupby("route_final")["address_present_gt"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "Address Present", "count": "Documents"})
    .reset_index()
)
address_by_route["Availability"] = address_by_route["Address Present"] / address_by_route["Documents"]
display(address_by_route.sort_values("Documents", ascending=False).style.format({"Availability": "{:.1%}"}).hide(axis="index"))

In [ ]:
plot_addr = address_by_route.sort_values("Availability")
plt.figure(figsize=(9.2, 6.0))
plt.barh(plot_addr["route_final"], 100 * plot_addr["Availability"], color=COLORS["teal"])
plt.xlabel("Rows with operationally valid Address (%)")
plt.title("Address Availability Differs by Document Route")
plt.xlim(0, 105)
plt.tight_layout()
plt.show()

**Key Finding**  
Dataset berisi multiple structural regimes dan field availability yang berbeda antar route.

**Pipeline Implication**  
Sistem menggunakan evidence driven routing dan specialist extraction. Stable routes dapat memakai parser khusus, sedangkan layout yang belum memiliki reliable structure masuk ke conservative fallback.

# 8 Name Complexity Requires More Than a Longest Text Heuristic

Name menjadi field yang sulit karena panjang karakter, jumlah token, dan struktur text berbeda antar dokumen. Length tetap berguna sebagai feature, tetapi tidak cukup sebagai rule tunggal.

In [ ]:
name_text = labeled["name_gt"].fillna("").astype(str).str.strip()
name_complexity = pd.DataFrame({
    "Character Length": name_text.str.len(),
    "Token Count": name_text.str.split().str.len(),
})
display(name_complexity.describe(percentiles=[0.25, 0.5, 0.75, 0.95]).round(2))

In [ ]:
plt.figure(figsize=(8.4, 4.8))
plt.hist(name_complexity["Character Length"], bins=25, color=COLORS["cyan"], edgecolor="white")
plt.xlabel("Ground truth Name character length")
plt.ylabel("Documents")
plt.title("Name Length Distribution")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8.4, 4.8))
plt.hist(name_complexity["Token Count"], bins=np.arange(0.5, name_complexity["Token Count"].max()+1.5, 1), color=COLORS["coral"], edgecolor="white")
plt.xlabel("Ground truth Name token count")
plt.ylabel("Documents")
plt.title("Name Token Count Distribution")
plt.tight_layout()
plt.show()

**Pipeline Implication**  
Final MyKad Name span ranker menggabungkan text shape, geometry, OCR confidence, dan relative position. Notebook tidak menyederhanakan proses menjadi rule seperti memilih line terpanjang.

# 9 Repeated Captures Preserve Complementary Evidence

Repeated captures hanya bermanfaat jika kualitas antar capture memang berbeda. Notebook mengukur within group spread untuk brightness, blur score, dan OCR confidence pada repeated groups.

In [ ]:
rep = labeled[labeled["identity_group"].isin(repeated_group_ids)].copy()
within_spread = (
    rep.groupby("identity_group")
    .agg(
        Captures=("filename", "size"),
        Brightness_Min=("mean_luminance", "min"),
        Brightness_Max=("mean_luminance", "max"),
        Blur_Min=("blur_score", "min"),
        Blur_Max=("blur_score", "max"),
        OCR_Conf_Min=("ocr_confidence", "min"),
        OCR_Conf_Max=("ocr_confidence", "max"),
    )
    .reset_index()
)
within_spread["Brightness Range"] = within_spread["Brightness_Max"] - within_spread["Brightness_Min"]
within_spread["Blur Range"] = within_spread["Blur_Max"] - within_spread["Blur_Min"]
within_spread["OCR Confidence Range"] = within_spread["OCR_Conf_Max"] - within_spread["OCR_Conf_Min"]
display(within_spread[["Captures", "Brightness Range", "Blur Range", "OCR Confidence Range"]].describe().round(3))

In [ ]:
plt.figure(figsize=(8.4, 4.8))
plt.hist(within_spread["Brightness Range"], bins=20, color=COLORS["orange"], edgecolor="white")
plt.xlabel("Brightness range inside repeated group")
plt.ylabel("Groups")
plt.title("Brightness Can Change Substantially Within the Same Label Derived Group")
plt.tight_layout()
plt.show()

### Same Group Visual Example

Satu label derived group dapat memiliki capture gelap, medium, dan sangat terang. OCR confidence juga tidak selalu bergerak searah dengan brightness.

In [ ]:
display(IPyImage(filename=str(ARTIFACT_DIR / "visuals" / "same_identity_variation.png"), width=1050))

**Key Finding**  
Repeated captures tidak identik dari sisi quality. Satu capture dapat membawa evidence yang lebih bersih daripada capture lain dari analytical group yang sama.

**Pipeline Implication**  
Repeated document reasoning layak dieksplorasi, tetapi cara menentukan peer harus dibedakan dengan jelas antara learned identity linking experiment dan label group assisted benchmark refinement.

# 10 OCR Text Contains Useful Identity Linking Signal

V3 mengeksplorasi Siamese identity linker. Sebelum membahas model tersebut, kita menguji pertanyaan yang lebih dasar: apakah OCR character patterns saja sudah membedakan same group pairs dari unrelated pairs?

In [ ]:
ocr_texts = labeled["ocr_text"].fillna("").astype(str)
tfidf = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    min_df=2,
    max_features=30000,
    lowercase=True,
    sublinear_tf=True,
)
X = tfidf.fit_transform(ocr_texts)
print("TF IDF matrix shape:", X.shape)
print("Vocabulary size     :", len(tfidf.vocabulary_))

In [ ]:
pair_rng = np.random.default_rng(SEED)
group_to_indices = {
    group: np.asarray(list(indices), dtype=int)
    for group, indices in labeled.groupby("identity_group").groups.items()
}

same_pairs = []
for indices in group_to_indices.values():
    if len(indices) < 2:
        continue
    pairs = list(combinations(indices, 2))
    if len(pairs) > 1000:
        chosen = pair_rng.choice(len(pairs), size=1000, replace=False)
        pairs = [pairs[i] for i in chosen]
    same_pairs.extend(pairs)

same_pairs = np.asarray(same_pairs, dtype=int)
different_pairs = []
all_idx = np.arange(len(labeled))
while len(different_pairs) < len(same_pairs):
    a = int(pair_rng.choice(all_idx))
    b = int(pair_rng.choice(all_idx))
    if a == b:
        continue
    if labeled.iloc[a]["identity_group"] == labeled.iloc[b]["identity_group"]:
        continue
    different_pairs.append((a, b))
different_pairs = np.asarray(different_pairs, dtype=int)

print("Same group pairs      :", len(same_pairs))
print("Different group pairs :", len(different_pairs))

In [ ]:
def sparse_pair_cosine(matrix, pairs, batch_size=4096):
    values = []
    for start in range(0, len(pairs), batch_size):
        batch = pairs[start:start+batch_size]
        left = matrix[batch[:, 0]]
        right = matrix[batch[:, 1]]
        values.extend(np.asarray(left.multiply(right).sum(axis=1)).ravel().tolist())
    return np.asarray(values)

same_sim = sparse_pair_cosine(X, same_pairs)
diff_sim = sparse_pair_cosine(X, different_pairs)
pair_auc = roc_auc_score(
    np.r_[np.ones(len(same_sim)), np.zeros(len(diff_sim))],
    np.r_[same_sim, diff_sim],
)

similarity_summary = pd.DataFrame({
    "Pair Type": ["Same label derived group", "Different group"],
    "Pairs": [len(same_sim), len(diff_sim)],
    "Median Cosine": [np.median(same_sim), np.median(diff_sim)],
    "Mean Cosine": [same_sim.mean(), diff_sim.mean()],
})
display(similarity_summary.style.format({"Median Cosine": "{:.3f}", "Mean Cosine": "{:.3f}"}).hide(axis="index"))
print(f"Diagnostic character TF IDF pair AUC: {pair_auc:.3f}")

In [ ]:
bins = np.linspace(0, 1, 32)
plt.figure(figsize=(8.4, 4.8))
plt.hist(diff_sim, bins=bins, density=True, alpha=0.55, color=COLORS["gray"], label="Different group")
plt.hist(same_sim, bins=bins, density=True, alpha=0.62, color=COLORS["blue"], label="Same label derived group")
plt.xlabel("Character TF IDF cosine similarity")
plt.ylabel("Density")
plt.title("OCR Similarity Separates Same Group Pairs From Unrelated Pairs")
plt.legend(frameon=False)
plt.tight_layout()
plt.show()

**Key Finding**  
Character level OCR similarity already contains useful identity signal with diagnostic AUC sekitar **0.918**, walaupun distributions masih overlap.

**Pipeline Implication**  
V3 mengembangkan signal ini melalui TF IDF, SVD, metadata features, dan Siamese representation dengan conservative precision first linking. Komponen tersebut adalah **development experiment**, bukan claim bahwa V28 memakai Siamese predicted clusters.

# 11 EDA Directly Determines the Pipeline Design

EDA bukan kumpulan plot yang berdiri sendiri. Tiga temuan utama secara langsung menentukan komponen arsitektur.

In [ ]:
synthesis = pd.DataFrame([
    ["Capture quality varies strongly", "OCR quality becomes inconsistent", "Quality aware preprocessing"],
    ["Document structure and field availability vary", "One universal parser fails", "Evidence driven routing and specialist extraction"],
    ["Repeated captures dominate many labeled images", "Random split overlap and uneven evidence", "Group aware component validation and repeated document analysis"],
], columns=["Data Evidence", "Technical Risk", "Pipeline Response"])
display(synthesis.style.hide(axis="index"))

> **Design Principle**  
> Pipeline complexity hanya dipertahankan ketika ada data pattern yang menjelaskan kenapa komponen tersebut diperlukan.

# 12 Quality Aware Preprocessing Adapts to the Capture

Final artifact menggunakan tiga candidate input: **Raw**, **Gray**, dan **Gray plus CLAHE**. Method dipilih berdasarkan frozen quality policy dari development run.

In [ ]:
policy = (
    labeled.groupby(["quality_group", "selected_preprocessing_method"])
    .size().reset_index(name="Documents")
    .sort_values(["quality_group", "Documents"], ascending=[True, False])
)
display(policy.style.hide(axis="index"))

In [ ]:
prep_counts = labeled["selected_preprocessing_method"].value_counts().rename_axis("Method").reset_index(name="Documents")
plt.figure(figsize=(7.6, 4.8))
prep_color_map = {"gray": COLORS["blue"], "gray_clahe": COLORS["green"], "raw": COLORS["orange"]}
plt.bar(prep_counts["Method"], prep_counts["Documents"], color=[prep_color_map.get(x, COLORS["gray"]) for x in prep_counts["Method"]])
plt.ylabel("Documents")
plt.title("Final Preprocessing Assignment")
plt.tight_layout()
plt.show()

### Visual Comparison on a Difficult Capture

Gambar berikut berasal dari original development notebook dan memperlihatkan Raw, Gray, serta Gray plus CLAHE pada satu dark capture. Visual ini digunakan sebagai **frozen development evidence**, bukan sebagai OCR benchmark baru.

In [ ]:
display(IPyImage(filename=str(ARTIFACT_DIR / "visuals" / "preprocessing_candidates.png"), width=980))

**Key Finding**  
Adaptive preprocessing mempertahankan detail original ketika enhancement tidak diperlukan dan meningkatkan local contrast pada degraded capture ketika dibutuhkan.

**Important Note**  
Notebook tidak menampilkan angka improvement preprocessing yang tidak tersedia pada retained artifact. Assignment counts di atas menunjukkan policy usage, bukan accuracy gain.

# 13 OCR Core Separates Text Detection and Text Recognition

Core OCR menggunakan **PP OCRv6 Medium** untuk text detection dan **SVTRv2 Server** untuk text recognition. Frozen artifact menyimpan recognized transcript, OCR confidence, geometry metadata, route, dan structured predictions.

In [ ]:
ocr_config = labeled[["ocr_model", "ocr_model_version", "ocr_runtime", "configuration"]].drop_duplicates().reset_index(drop=True)
display(ocr_config.style.hide(axis="index"))

### Example Identity Card and Frozen OCR Transcript

Contoh berikut menunjukkan satu MyKad dari original development evidence bersama transcript final OCR. Notebook hanya menampilkan frozen evidence sehingga tidak ada OCR engine pengganti yang dijalankan.

In [ ]:
display(IPyImage(filename=str(ARTIFACT_DIR / "visuals" / "ocr_transcript_hamzah.png"), width=1050))

### Frozen OCR Bounding Box Evidence

Visual berikut menunjukkan OCR line evidence pada MyKad yang sama. Bounding boxes berasal dari frozen development evidence. Mereka tidak digambar ulang secara spekulatif pada notebook ini.

In [ ]:
display(IPyImage(filename=str(ARTIFACT_DIR / "visuals" / "ocr_bbox_evidence.png"), width=820))

In [ ]:
hamzah = labeled[labeled["filename"].eq("image_001.jpg")].iloc[0]
trace = pd.DataFrame([
    ["Quality group", hamzah["quality_group"]],
    ["Selected preprocessing", hamzah["selected_preprocessing_method"]],
    ["Document family", hamzah["document_family"]],
    ["OCR confidence", hamzah["ocr_confidence"]],
    ["Final Name", hamzah["name_pred"]],
    ["Final DOB", hamzah["birth_date_pred"]],
    ["Final Address", hamzah["address_pred"]],
], columns=["Trace Item", "Value"])
display(trace.style.hide(axis="index"))

**Key Finding**  
OCR menyediakan text evidence, confidence, dan geometry. Structured KYC output belum selesai pada tahap ini karena line yang sama dapat memiliki arti berbeda pada document route yang berbeda.

**Pipeline Response**  
OCR evidence diteruskan ke routing dan specialist extraction.

# 14 Route First Then Extract With Specialist Logic

Routing menggunakan OCR text, layout cues, MRZ evidence, field numbering, dan route specific patterns. Notebook sengaja tidak menyebut seluruh routing sebagai satu learned multiclass classifier karena source menunjukkan kombinasi learned and rule based evidence.

In [ ]:
specialists = pd.DataFrame([
    ["MyKad", "Name candidate ranking, DOB interpretation, Address line reconstruction"],
    ["MRZ and Passport", "MRZ structure, parsing, checksum and structured decoding"],
    ["EU Driving Licence", "Numbered field interpretation"],
    ["Country Specific IDs", "Keyword, layout, and document pattern rules"],
    ["Unknown Route", "Conservative generic fallback"],
], columns=["Route Family", "Specialist Logic"])
display(specialists.style.hide(axis="index"))

### Why Routing Matters

Route specific performance is not uniform. The following frozen benchmark diagnostic shows where stable document structure helps and where the unknown layout tail remains difficult.

In [ ]:
route_perf = (
    labeled.groupby("route_final")
    .agg(
        Documents=("filename", "size"),
        Name_EM=("name_exact", "mean"),
        Name_Similarity=("name_similarity", "mean"),
        DOB_EM=("dob_exact", "mean"),
    )
    .reset_index()
)
display(route_perf.sort_values("Documents", ascending=False).style.format({"Name_EM": "{:.1%}", "Name_Similarity": "{:.1%}", "DOB_EM": "{:.1%}"}).hide(axis="index"))

In [ ]:
plot_perf = route_perf.sort_values("Name_Similarity")
plt.figure(figsize=(9.3, 6.1))
colors = [COLORS["coral"] if x == "Unknown Route" else COLORS["blue"] for x in plot_perf["route_final"]]
plt.barh(plot_perf["route_final"], 100 * plot_perf["Name_Similarity"], color=colors)
plt.xlabel("Mean Name normalized similarity (%)")
plt.title("Name Similarity Differs Strongly by Document Route")
plt.tight_layout()
plt.show()

# 15 The System Evolved Through Error Driven Development

Final deliverable bukan satu model yang langsung muncul dalam satu training run. Source V3 sampai V27 menunjukkan iterative development yang berfokus pada failure mode berbeda.

Penting untuk membedakan **development journey** dari **final computation flow**. Beberapa eksperimen diuji tetapi tidak dipakai sebagai stage linear pada final V28 benchmark.

In [ ]:
display(development_timeline.style.hide(axis="index"))

### Retained Core Components

- quality aware preprocessing
- PP OCRv6 text detection
- SVTRv2 recognition
- document routing
- specialist field extraction
- field specific refinements such as MyKad Name span ranking

### Development Experiments

- visual identity signals
- face and graph identity branches
- Siamese identity linking

### Downstream Benchmark Refinements

- repeated document peer evidence on known labeled groups
- V27 fuzzy medoid refinement
- V28 Generic ID refinement

# 16 Siamese Identity Linking Was a Development Experiment

V3 source benar benar membangun representation pipeline untuk repeated capture linking. Namun notebook final tidak mengklaim bahwa V28 memakai Siamese predicted clusters.

In [ ]:
v3_source = (ARTIFACT_DIR / "scripts_v3" / "01_ppocrv5_identity_consensus.py").read_text(encoding="utf-8", errors="ignore")
architecture_evidence = pd.DataFrame([
    ["Text representation", "Character TF IDF with 2 to 5 grams"],
    ["Dimensionality reduction", "Truncated SVD up to 96 dimensions"],
    ["Additional evidence", "OCR confidence, item count, field presence, quality and document family features"],
    ["Siamese encoder", "256 to 128 to 64 normalized representation"],
    ["Similarity", "Cosine similarity"],
    ["Linking policy", "Precision first threshold with conservative clustering"],
], columns=["Component", "Source Backed Design"])
display(architecture_evidence.style.hide(axis="index"))

assert "class SiameseLinker" in v3_source
print("Source check: SiameseLinker implementation found")

### Later Peer Consensus Is a Separate Benchmark Refinement

Later V18 to V28 benchmark refinements use known **label derived identity groups** when aggregating peer evidence on labeled data. Karena peer membership tersebut tidak berasal dari Siamese inference, hasil V28 dipresentasikan sebagai **labeled benchmark refinement**, bukan unseen identity deployment result.

# 17 Validation Must Match the Question Being Asked

Tidak semua stage memakai validation design yang sama. Notebook memisahkan tiga konteks:

1. **Component validation** dapat menggunakan identity disjoint groups ketika supervised training membutuhkan train validation separation.
2. **V28 downstream holdout** memakai calibration and holdout logic pada refinement stage dan tidak disebut sebagai identity disjoint evaluation.
3. **Final V28 labeled benchmark** merangkum performa pada 632 labeled rows.

In [ ]:
validation_summary = pd.DataFrame([
    ["Supervised development components", "Identity disjoint where implemented", "Measure component generalization"],
    ["V28 downstream refinement", "Document based calibration and holdout", "Gate the downstream Generic ID refinement"],
    ["Final V28 labeled benchmark", "632 labeled benchmark rows", "Report final released labeled performance"],
], columns=["Evaluation Context", "Design", "Purpose"])
display(validation_summary.style.hide(axis="index"))

# 18 Final V28 Reaches 66.77 Percent Macro Field Exact Match

Final evaluation menghitung ulang metric dari released prediction table. Tidak ada metric utama yang ditulis manual.

In [ ]:
def normalize_text(value):
    if pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value).strip().upper())

def exact_rate(gt_col, pred_col):
    gt = labeled[gt_col].map(normalize_text)
    pred = labeled[pred_col].map(normalize_text)
    return float((gt == pred).mean())

name_em = exact_rate("name_gt", "name_pred")
dob_em = exact_rate("birth_date_gt", "birth_date_pred")
address_em = exact_rate("address_gt", "address_pred")
macro_em = (name_em + dob_em + address_em) / 3
full_record = float(labeled["full_record_correct"].astype(bool).mean())
normalized_similarity = float(labeled["normalized_edit_similarity"].mean())
record_cer = float(labeled["record_cer"].mean())

metrics = pd.DataFrame([
    ["Name Exact Match", name_em],
    ["DOB Exact Match", dob_em],
    ["Address Exact Match", address_em],
    ["Macro Field Exact Match", macro_em],
    ["Full Record Exact Match", full_record],
    ["Normalized Similarity", normalized_similarity],
    ["Record CER", record_cer],
], columns=["Metric", "Value"])

display(metrics.style.format({"Value": lambda x: f"{x:.2%}" if x <= 1 else f"{x:.4f}"}).hide(axis="index"))

In [ ]:
# Strong integrity checks against released metrics.json
checks = {
    "Name EM": name_em,
    "DOB EM": dob_em,
    "Address EM": address_em,
    "Macro Field EM": macro_em,
    "Full Record Accuracy": full_record,
    "Normalized Edit Similarity": normalized_similarity,
    "CER": record_cer,
}
for key, computed in checks.items():
    reference = float(v28_metrics_ref[key])
    assert abs(computed - reference) < 1e-10, (key, computed, reference)
print("Metric integrity check: PASS")

In [ ]:
field_plot = pd.DataFrame({
    "Field": ["Name", "DOB", "Address"],
    "Exact Match": [name_em, dob_em, address_em],
})
plt.figure(figsize=(8.0, 4.8))
bar_colors = [COLORS["blue"], COLORS["orange"], COLORS["green"]]
bars = plt.bar(field_plot["Field"], 100 * field_plot["Exact Match"], color=bar_colors, width=0.62)
plt.ylabel("Exact Match (%)")
plt.ylim(0, 100)
plt.title("Exact Match Differs Strongly by Target Field")
for bar, value in zip(bars, field_plot["Exact Match"]):
    plt.text(bar.get_x()+bar.get_width()/2, 100*value+2, f"{100*value:.2f}%", ha="center", fontsize=10)
plt.tight_layout()
plt.show()

## Address Metric Requires a Blank Label Caveat

All row Address Exact Match terlihat tinggi karena mayoritas rows memang tidak memiliki operationally valid Address ground truth. Karena itu, present only evaluation perlu dihitung terpisah.

In [ ]:
address_present = labeled[labeled["address_present_gt"].astype(bool)].copy()
present_address_em = float(
    (address_present["address_gt"].map(normalize_text) == address_present["address_pred"].map(normalize_text)).mean()
)

address_interpretation = pd.DataFrame([
    ["All rows", len(labeled), address_em],
    ["Address present only", len(address_present), present_address_em],
], columns=["Evaluation Scope", "Rows", "Address Exact Match"])
display(address_interpretation.style.format({"Address Exact Match": "{:.2%}"}).hide(axis="index"))

**Technical Interpretation**  
Address all row EM sebesar **83.70%** bersifat blank sensitive. Pada 112 rows dengan operationally valid Address ground truth, present only Exact Match sekitar **25%**. Karena itu, Address tidak boleh disebut sebagai field yang sudah solved hanya berdasarkan all row score.

## V28 Is an Incremental Downstream Improvement

V28 tidak mengganti OCR backbone. Improvement terjadi pada downstream Generic ID refinement.

In [ ]:
def core_metrics(frame):
    lab = frame[frame["has_ground_truth"].astype(bool)].copy()
    return {
        "Macro Field EM": float(lab["macro_field_exact"].mean()),
        "DOB EM": float(lab["dob_exact"].mean()),
        "Normalized Similarity": float(lab["normalized_edit_similarity"].mean()),
        "Record CER": float(lab["record_cer"].mean()),
    }

v27_summary = core_metrics(v27)
v28_summary = core_metrics(v28)
comparison = pd.DataFrame([
    [metric, v27_summary[metric], v28_summary[metric], v28_summary[metric] - v27_summary[metric]]
    for metric in v27_summary
], columns=["Metric", "V27", "V28", "Change"])
display(comparison.style.format({"V27": "{:.4f}", "V28": "{:.4f}", "Change": "{:+.4f}"}).hide(axis="index"))

# 19 Unknown Route Name Extraction Remains the Primary Bottleneck

Global score menyembunyikan perbedaan besar antar route. Error analysis menunjukkan bahwa Name extraction pada Unknown Route merupakan area yang paling lemah.

In [ ]:
error_route = (
    labeled.groupby("route_final")
    .agg(
        Documents=("filename", "size"),
        Name_EM=("name_exact", "mean"),
        Name_Similarity=("name_similarity", "mean"),
        DOB_EM=("dob_exact", "mean"),
    )
    .reset_index()
    .sort_values("Name_Similarity")
)
display(error_route.style.format({"Name_EM": "{:.2%}", "Name_Similarity": "{:.2%}", "DOB_EM": "{:.2%}"}).hide(axis="index"))

In [ ]:
plt.figure(figsize=(9.2, 6.1))
colors = [COLORS["coral"] if x == "Unknown Route" else COLORS["teal"] for x in error_route["route_final"]]
plt.barh(error_route["route_final"], 100 * error_route["Name_Similarity"], color=colors)
plt.xlabel("Mean Name normalized similarity (%)")
plt.title("Unknown Route Forms the Largest Name Extraction Bottleneck")
plt.tight_layout()
plt.show()

### Representative Failure Cases

Frozen development examples memperlihatkan bahwa failure tidak hanya berasal dari satu penyebab. OCR recognition error, unknown layout, field confusion, dan insufficient evidence dapat menghasilkan error yang berbeda.

In [ ]:
display(IPyImage(filename=str(ARTIFACT_DIR / "visuals" / "error_cases.png"), width=1000))

**Improvement Priority**  
Jika pipeline dilanjutkan setelah kompetisi, prioritas tertinggi adalah memperluas reliable layout routing dan memperkuat Name candidate extraction pada unknown generic documents.

# 20 Structured Fields Remain Traceable to OCR Evidence

Untuk workflow KYC, model score saja tidak cukup. Sistem perlu menyimpan evidence yang menjelaskan bagaimana structured field dihasilkan.

In [ ]:
traceability = pd.DataFrame([
    ["Image evidence", "Frozen document capture and localization metadata"],
    ["OCR evidence", "Recognized text, confidence, and line geometry from development artifacts"],
    ["Route evidence", "Document family and specialist route"],
    ["Field evidence", "Selected nodes, parsing logic, and final structured fields"],
    ["Evaluation evidence", "Ground truth comparison on labeled benchmark"],
], columns=["Trace Layer", "Available Evidence"])
display(traceability.style.hide(axis="index"))

> **Interpretation**  
> Notebook menggunakan istilah **traceable** atau **auditable**, bukan claim bahwa seluruh internal neural representation dapat dijelaskan secara penuh.

# 21 Confidence Gated Automation Is Safer Than Fully Autonomous KYC

Full Record Exact Match masih **37.66%** dan Unknown Route Name extraction tetap menjadi bottleneck. Karena itu, hasil project lebih cocok digunakan untuk **confidence gated automation** daripada keputusan KYC yang sepenuhnya autonomous.

In [ ]:
recommendations = pd.DataFrame([
    ["Clear structured output", "High confidence, stable route, complete fields", "Straight through extraction"],
    ["Uncertain structured output", "Low confidence, unknown route, missing or inconsistent fields", "Human review"],
    ["Operational monitoring", "Route mix, confidence distribution, failure rate", "Monitor by route and field"],
    ["Auditability", "OCR evidence and route decision retained", "Support review and correction"],
], columns=["Workflow State", "Decision Evidence", "Recommended Action"])
display(recommendations.style.hide(axis="index"))

### Proposed KYC Flow

`ID Upload → Quality Check → OCR and Structured Extraction → Confidence and Consistency Gate → Auto Process or Human Review`

Business value tidak hanya berasal dari mengganti seluruh pekerjaan manual. Pipeline dapat mengurangi beban human verification dengan memprioritaskan review pada cases yang paling uncertain.

# 22 Competition Test Output Uses the Retained Official Predictions

Later V28 labeled benchmark refinements tidak digunakan untuk overwrite retained official test predictions karena peer group provenance berbeda dengan clean test inference. Notebook sengaja memisahkan kedua tables untuk mencegah accidental export.

In [ ]:
assert len(official_test) == 100
assert official_test["filename"].is_unique
assert set(official_test.columns) == {"filename", "name", "birth_date", "address"}

prediction_path = OUTPUT_DIR / "prediction.csv"
official_test.to_csv(prediction_path, index=False)

print("Competition Output Check")
print("Rows              :", len(official_test))
print("Unique filenames  :", official_test["filename"].nunique())
print("Output file       :", prediction_path)
print("Status            : PASS")
display(official_test.head(8))

# 23 What This Notebook Demonstrates and What It Does Not

### Demonstrated

- downstream artifact reproducibility
- rich EDA connected directly to design choices
- traceable OCR and route evidence
- specialist structured extraction methodology
- exact reproduction of final V28 labeled benchmark metrics
- safe regeneration of retained official competition predictions

### Not Demonstrated

- raw OCR retraining inside this notebook
- full 732 image OCR inference from scratch during Run All
- V28 as an unseen identity deployment score
- Siamese predicted clusters as the peer source used by final V28
- fully autonomous KYC readiness

# 24 Conclusion

Heterogeneity pada dataset muncul pada tiga level: **capture quality**, **document structure**, dan **repeated captures**. EDA menunjukkan bahwa ketiga masalah tersebut membutuhkan respons teknis yang berbeda.

Pipeline kemudian dibangun sebagai modular system yang menggabungkan:

1. quality aware preprocessing,
2. PP OCRv6 text detection,
3. SVTRv2 text recognition,
4. evidence driven document routing,
5. specialist field extraction,
6. field specific downstream refinement.

Final V28 labeled benchmark mencapai **66.77% Macro Field Exact Match**. Hasil tersebut cukup kuat sebagai structured extraction baseline, tetapi Name extraction pada Unknown Route dan Address present only performance masih menjadi limitation penting.

Untuk dunia nyata, rekomendasi utama adalah **confidence gated KYC workflow** dengan human review pada low confidence dan structurally uncertain cases.

# 25 Final Notebook Quality Check

Cell terakhir memastikan final notebook konsisten dengan angka benchmark dan competition output yang sudah ditetapkan.

In [ ]:
assert len(labeled) == 632
assert len(test_artifact_rows) == 100
assert len(official_test) == 100
assert abs(macro_em - 0.6677) < 5e-4
assert abs(name_em - 0.4620) < 5e-4
assert abs(dob_em - 0.7041) < 5e-4
assert abs(address_em - 0.8370) < 5e-4
assert 0.24 <= present_address_em <= 0.26
assert 0.90 <= leakage_trials.mean() <= 0.93
assert 0.91 <= pair_auc <= 0.93

print("FINAL QA")
print("Notebook evidence integrity : PASS")
print("Benchmark metric integrity  : PASS")
print("Competition output integrity: PASS")
print("Ready for final review")